# 新闻分类：多分类问题示例

对应 Keras 版本：`Keras应用/第四章-神经网络入门：分类与回归/Reuters新闻多分类.ipynb`

使用 PyTorch 完成 Reuters 新闻主题分类：
- 输入：新闻标题（编码为整数序列）
- 输出：46 个主题之一
- 类别数：46 类

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt

print(f"PyTorch: {torch.__version__}")

## 1. Reuters 数据集

In [ ]:
from tensorflow.keras.datasets import reuters

(train_data, train_labels), (test_data, test_labels) = reuters.load_data(num_words=10000)

print(f"训练集样本数: {len(train_data)}")
print(f"测试集样本数: {len(test_data)}")
print(f"类别数: {max(train_labels) + 1}")
print(f"\n第一条训练样本: {train_data[0][:20]}...")
print(f"第一条训练样本标签: {train_labels[0]}")

## 2. 数据预处理：Multi-hot 编码

In [ ]:
def vectorize_sequences(sequences, dimension=10000):
    results = np.zeros((len(sequences), dimension), dtype=np.float32)
    for i, sequence in enumerate(sequences):
        results[i, sequence] = 1.
    return results

x_train = torch.from_numpy(vectorize_sequences(train_data))
x_test = torch.from_numpy(vectorize_sequences(test_data))
y_train = torch.tensor(train_labels, dtype=torch.long)
y_test = torch.tensor(test_labels, dtype=torch.long)

print(f"x_train 形状: {x_train.shape}")
print(f"y_train 形状: {y_train.shape}")

## 3. 构建模型

对应 Keras 版本的结构：
```
Input(10000) → Dense(64, relu) → Dense(64, relu) → Dense(46)
```

**注意**：PyTorch 的 `CrossEntropyLoss` 内部已包含 softmax，所以输出层不需要激活函数。

In [ ]:
class ReutersClassifier(nn.Module):
    """
    Reuters 46 类新闻分类模型。
    结构对应 Keras 版本：Dense(64) → Dense(64) → Dense(46)
    """
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(10000, 64)
        self.fc2 = nn.Linear(64, 64)
        self.fc3 = nn.Linear(64, 46)
    
    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        return self.fc3(x)  # 输出 logits（CrossEntropyLoss 内置 softmax）

model = ReutersClassifier()
print(model)
print(f"\n总参数: {sum(p.numel() for p in model.parameters())}")

## 4. 划分验证集

In [ ]:
# 划分验证集（前 1000 条）
x_val = x_train[:1000]
y_val = y_train[:1000]
x_train_partial = x_train[1000:]
y_train_partial = y_train[1000:]

print(f"训练集: {x_train_partial.shape[0]} 样本")
print(f"验证集: {x_val.shape[0]} 样本")

## 5. 训练循环

In [ ]:
def train_epoch(model, x, y, optimizer, loss_fn, batch_size=512):
    model.train()
    total_loss = 0
    correct = 0
    n = len(x)
    indices = torch.randperm(n)
    x, y = x[indices], y[indices]
    
    for i in range(0, n, batch_size):
        bx, by = x[i:i+batch_size], y[i:i+batch_size]
        optimizer.zero_grad()
        out = model(bx)
        loss = loss_fn(out, by)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * bx.size(0)
        correct += (out.argmax(dim=1) == by).sum().item()
    return total_loss / n, correct / n


@torch.no_grad()
def evaluate(model, x, y, loss_fn):
    model.eval()
    out = model(x)
    loss = loss_fn(out, y)
    correct = (out.argmax(dim=1) == y).sum().item()
    return loss.item(), correct / len(y)


# 训练
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
loss_fn = nn.CrossEntropyLoss()

epochs = 20
history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}

for epoch in range(epochs):
    train_loss, train_acc = train_epoch(model, x_train_partial, y_train_partial,
                                        optimizer, loss_fn)
    val_loss, val_acc = evaluate(model, x_val, y_val, loss_fn)
    
    history["train_loss"].append(train_loss)
    history["train_acc"].append(train_acc)
    history["val_loss"].append(val_loss)
    history["val_acc"].append(val_acc)
    
    print(f"Epoch {epoch+1:2d}: train_loss={train_loss:.4f}, val_loss={val_loss:.4f}, "
          f"val_acc={val_acc:.4f}")

## 6. 绘制训练曲线

In [ ]:
epochs_range = range(1, epochs + 1)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(epochs_range, history["train_loss"], "bo", label="Training loss")
axes[0].plot(epochs_range, history["val_loss"], "b", label="Validation loss")
axes[0].set_xlabel("Epochs")
axes[0].set_ylabel("Loss")
axes[0].legend()
axes[0].grid(True)

axes[1].plot(epochs_range, history["train_acc"], "bo", label="Training acc")
axes[1].plot(epochs_range, history["val_acc"], "b", label="Validation acc")
axes[1].set_xlabel("Epochs")
axes[1].set_ylabel("Accuracy")
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.show()

## 7. 测试集评估

In [ ]:
test_loss, test_acc = evaluate(model, x_test, y_test, loss_fn)
print(f"测试损失: {test_loss:.4f}")
print(f"测试准确率: {test_acc:.4f}")

## 8. 预测

In [ ]:
with torch.no_grad():
    predictions = model(x_test[:10])
    probs = F.softmax(predictions, dim=1)
    pred_classes = predictions.argmax(dim=1)

print(f"{'样本':>6} | {'预测类别':>6} | {'概率':>8} | {'真实标签':>6}")
print("-" * 42)
for i in range(10):
    match = "✓" if pred_classes[i] == y_test[i] else "✗"
    print(f"  #{i:>4} | {pred_classes[i]:>6} | {probs[i, pred_classes[i]].item():>8.4f} | {y_test[i]:>6} {match}")

### 多分类 vs 二分类 对比

| 维度 | 二分类 (IMDB) | 多分类 (Reuters) |
|------|---------------|-----------------|
| 输出层 | `Dense(1)` + `sigmoid` | `Dense(46)` (无激活) |
| 损失函数 | `nn.BCELoss()` | `nn.CrossEntropyLoss()` |
| 标签类型 | `float32` (0/1) | `long` (0~45) |
| 预测 | `output >= 0.5` | `output.argmax(dim=1)` |
| 概率 | `torch.sigmoid()` | `F.softmax(dim=1)` |